In [ ]:
import os
import time
import pandas as pd
from PIL import Image
from tqdm import tqdm
import google.generativeai as genai

# ==========================================
# 1. KONFIGURASI API & MODEL
# ==========================================
# Masukkan API KEY Google AI Studio milikmu
API_KEY = "#Insert_Your_API_Key_Here#" 
genai.configure(api_key=API_KEY)

# Berdasarkan kuotamu, Gemma 3 punya jatah 14.4K RPD (Sangat Aman!)
MODEL_NAME = 'models/gemma-3-27b-it' 

def extract_with_gemma(model, img_path):
    """Fungsi untuk mengirim gambar ke API Gemma 3 dan mengambil teks tabelnya."""
    try:
        image = Image.open(img_path).convert("RGB")
        prompt = (
            "You are a data analyst. Extract the main features, overall trends, and key comparisons from this image. Do NOT output a table, lists, or markdown. Write a single, highly condensed summary in natural language. STRICT LIMIT: Maximum 45 words."
        )
        # Mengirim request ke API
        response = model.generate_content([prompt, image])
        # Membersihkan newline agar rapi saat disimpan di CSV
        return response.text.replace('\n', ' | ').strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# ==========================================
# 2. MAIN EXECUTION
# ==========================================
def main():
    # Sesuaikan dengan path folder skripsimu
    CSV_INPUT = "../data.csv" 
    IMAGE_DIR = "../images"
    CSV_OUTPUT = "data_short_gemma.csv"

    # Load dataset asli
    try:
        df = pd.read_csv(CSV_INPUT, encoding='latin-1') 
        print(f"✅ Dataset dimuat: {len(df)} baris.")
    except Exception as e:
        print(f"❌ Gagal memuat CSV: {e}")
        return

    # Inisialisasi Model Gemma 3
    print(f"⏳ Menghubungkan ke API {MODEL_NAME}...")
    model = genai.GenerativeModel(MODEL_NAME)

    # IDE JENIUS: Ambil gambar unik berdasarkan URL di kolom 'graph'
    unique_df = df.drop_duplicates(subset=['graph'])
    print(f"🔥 BINGO! Dari {len(df)} data, hanya ada {len(unique_df)} gambar unik.")

    extracted_dict = {}

    # Proses ekstraksi hanya untuk gambar yang unik
    print("\n🚀 Memulai Ekstraksi (Gemma 3 API Mode)...")
    for idx, row in tqdm(unique_df.iterrows(), total=len(unique_df), desc="Extracting"):
        # Nama file sesuai index di folder images_cache kamu
        img_filename = f"{idx}.jpg"
        img_path = os.path.join(IMAGE_DIR, img_filename)
        url = row['graph']
        
        if os.path.exists(img_path):
            # Percobaan ekstraksi
            result = extract_with_gemma(model, img_path)
            extracted_dict[url] = result
            
            # Gemma 3 punya RPM 30, jadi jeda 2-3 detik sudah sangat aman
            time.sleep(3) 
        else:
            print(f"⚠️ Gambar tidak ditemukan: {img_filename}")
            extracted_dict[url] = ""

    # 3. MAPPING KEMBALI KE DATASET UTAMA
    print("\n🔄 Memetakan hasil ke seluruh baris dataset...")
    df['description'] = df['graph'].map(extracted_dict)

    # Simpan hasil akhir
    df.to_csv(CSV_OUTPUT, index=False)
    print(f"🎉 SELESAI! Hasil ekstraksi disimpan ke: {CSV_OUTPUT}")

if __name__ == "__main__":
    main()

c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\gabyg\AppData\Local\Temp\ipykernel_43488\469769730.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more det

✅ Dataset dimuat: 1054 baris.
⏳ Menghubungkan ke API models/gemma-3-27b-it...
🔥 BINGO! Dari 1054 data, hanya ada 125 gambar unik.

🚀 Memulai Ekstraksi (Gemma 3 API Mode)...


Extracting: 100%|██████████| 125/125 [16:41<00:00,  8.01s/it]



🔄 Memetakan hasil ke seluruh baris dataset...
🎉 SELESAI! Hasil ekstraksi disimpan ke: data_short_gemma.csv


In [ ]:
import os, time, json
import pandas as pd
from PIL import Image
from tqdm import tqdm
import google.generativeai as genai

API_KEY = "#Insert_Your_API_Key_Here#"
genai.configure(api_key=API_KEY)

MODEL_NAME = 'models/gemma-3-27b-it'
CACHE_FILE = "extraction_cache.json"
PROMPT = (
    "You are a data analyst. Extract the main features, overall trends, "
    "and key comparisons from this image. Do NOT output a table, lists, or markdown. "
    "Write a single, highly condensed summary in natural language. "
    "STRICT LIMIT: Maximum 45 words."
)

generation_config = genai.GenerationConfig(
    temperature=0.001,
    top_p=1.0,
    top_k=1,
    max_output_tokens=100,
    candidate_count=1,
)

def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_cache(cache):
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

def extract_with_gemma(model, img_path):
    try:
        image = Image.open(img_path).convert("RGB")
        response = model.generate_content([PROMPT, image])
        return response.text.replace('\n', ' | ').strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

def main():
    CSV_INPUT = "../data.csv"
    IMAGE_DIR = "../images"
    CSV_OUTPUT = "data_short_gemma2.csv"

    df = pd.read_csv(CSV_INPUT, encoding='latin-1')
    print(f"✅ Dataset dimuat: {len(df)} baris.")

    model = genai.GenerativeModel(MODEL_NAME, generation_config=generation_config)
    extracted_dict = load_cache()  # Resume dari cache jika ada

    unique_df = df.drop_duplicates(subset=['graph'])
    print(f"🔥 {len(unique_df)} gambar unik, {len(extracted_dict)} sudah di-cache.")

    for idx, row in tqdm(unique_df.iterrows(), total=len(unique_df)):
        url = row['graph']
        if url in extracted_dict:  # Skip jika sudah ada
            continue

        img_path = os.path.join(IMAGE_DIR, f"{idx}.jpg")
        if os.path.exists(img_path):
            extracted_dict[url] = extract_with_gemma(model, img_path)
            save_cache(extracted_dict)
            time.sleep(3)
        else:
            extracted_dict[url] = ""

    df['description'] = df['graph'].map(extracted_dict)
    
    # Simpan metadata eksperimen
    pd.DataFrame([{
        "model": MODEL_NAME,
        "temperature": 0.0,
        "run_date": pd.Timestamp.now().isoformat(),
    }]).to_csv("run_metadata.csv", index=False)
    
    df.to_csv(CSV_OUTPUT, index=False)
    print(f"🎉 Selesai! Hasil: {CSV_OUTPUT}")

if __name__ == "__main__":
    main()

✅ Dataset dimuat: 1054 baris.
🔥 125 gambar unik, 0 sudah di-cache.


100%|██████████| 125/125 [16:07<00:00,  7.74s/it]


🎉 Selesai! Hasil: data_short_gemma2.csv
